Environment & Load Model GRPO

In [2]:
!pip install -q "langchain>=0.3,<0.4" "langchain-community>=0.3,<0.4" "langchain-text-splitters>=0.3,<0.4" "langchain-huggingface>=0.1,<0.4" faiss-cpu rank_bm25 pypdf sentence-transformers ddgs duckduckgo-search
!pip install -q --upgrade unsloth unsloth_zoo

import os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from unsloth import FastLanguageModel   # import unsloth lebih dulu
import torch, re

SFT_REPO = "putraharifin/llama-3.2-3b-legal-id-PGABL-putraharifin25"
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=SFT_REPO,
    max_seq_length=4096,
    dtype=None,
    load_in_4bit=True,
    token=os.environ["HF_TOKEN"],
)
FastLanguageModel.for_inference(model)

def llm_generate(messages, max_new_tokens=512, temperature=0.3, top_p=0.9, n=1):
    """Generate dari chat messages. Mengembalikan list string sepanjang n."""
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False).to("cuda")
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            num_return_sequences=n,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    plen = inputs["input_ids"].shape[1]
    return [tokenizer.decode(o[plen:], skip_special_tokens=True).strip() for o in out]

print("Model siap. VRAM:", f"{torch.cuda.memory_reserved()/1024**3:.2f} GB")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.10.7: Fast Llama patching. Transformers: 5.16.1.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


Unsloth: Not splitting across GPUs; the model fits on cuda:0 with room to train. Pass device_map = "unsloth_balanced" to split it anyway.


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load putraharifin/llama-3.2-3b-legal-id-PGABL-putraharifin25 as a legacy tokenizer.


Model siap. VRAM: 2.13 GB


Loading PDF, Parent-Child Splitting, Metadata Enrichment

In [4]:
import glob, uuid
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

REQUIRED_PDFS = [
    "PP Nomor 5 Tahun 2021.pdf",
    "PP Nomor 35 Tahun 2021.pdf",
    "PP Nomor 51 Tahun 2023.pdf",
    "UU Nomor 6 Tahun 2023.pdf",
]

def find_pdf(name):
    for root in ["/kaggle/input", "/kaggle/working", "."]:
        hits = glob.glob(os.path.join(root, "**", name), recursive=True)
        if hits:
            return hits[0]
    raise FileNotFoundError(f"'{name}' tidak ditemukan. Upload sebagai Kaggle Dataset (Add Input) atau ke /kaggle/working.")

# --- Load per halaman + Metadata Enrichment ---
page_docs = []
for fname in REQUIRED_PDFS:
    path = find_pdf(fname)
    pages = PyPDFLoader(path).load()
    m = re.match(r"^(UU|PP) Nomor (\d+) Tahun (\d{4})", fname)
    base = fname.rsplit(".pdf", 1)[0]
    kept = []
    for d in pages:
        if not d.page_content.strip():
            continue
        pg = int(d.metadata.get("page", 0)) + 1
        d.metadata = {
            "source": fname,                         # nama file asli
            "page": pg,                              # nomor halaman (1-based)
            "citation": f"{base} halaman {pg}",      # untuk sitasi otomatis
            "jenis": m.group(1),                     # UU / PP
            "nomor": int(m.group(2)),
            "tahun": int(m.group(3)),
        }
        kept.append(d)
    page_docs.extend(kept)
    print(f"{fname}: {len(pages)} halaman, {len(kept)} berisi teks")

# --- Parent chunks (konteks LLM) ---
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
parent_docs = parent_splitter.split_documents(page_docs)
for p in parent_docs:
    p.metadata["parent_id"] = str(uuid.uuid4())

# --- Child chunks (pencarian vektor); mewarisi metadata + parent_id ---
child_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=40)
child_docs = []
for p in parent_docs:
    child_docs.extend(child_splitter.split_documents([p]))

print(f"\nHalaman: {len(page_docs)} | Parent: {len(parent_docs)} | Child: {len(child_docs)}")
print("Contoh metadata child:", child_docs[0].metadata)

PP Nomor 5 Tahun 2021.pdf: 739 halaman, 739 berisi teks
PP Nomor 35 Tahun 2021.pdf: 56 halaman, 56 berisi teks
PP Nomor 51 Tahun 2023.pdf: 27 halaman, 27 berisi teks
UU Nomor 6 Tahun 2023.pdf: 1127 halaman, 1127 berisi teks

Halaman: 1949 | Parent: 3219 | Child: 13364
Contoh metadata child: {'source': 'PP Nomor 5 Tahun 2021.pdf', 'page': 1, 'citation': 'PP Nomor 5 Tahun 2021 halaman 1', 'jenis': 'PP', 'nomor': 5, 'tahun': 2021, 'parent_id': '730c5889-40be-45cf-a884-19efdb2de86c'}


Vector DB, InMemoryStore & Parent-Child Retriever

In [5]:
from typing import Any, List
from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever
try:
    from langchain_core.stores import InMemoryStore
except ImportError:
    from langchain.storage import InMemoryStore
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",  # open-source
    model_kwargs={"device": "cuda"},
    encode_kwargs={"batch_size": 64, "normalize_embeddings": True},
)

# Vector DB lokal (hanya child chunks yang di-embed)
vectorstore = FAISS.from_documents(child_docs, embeddings)

# InMemoryStore: parent_id -> Parent chunk
docstore = InMemoryStore()
docstore.mset([(p.metadata["parent_id"], p) for p in parent_docs])

class ParentChildRetriever(BaseRetriever):
    """Cari Child Chunk di vektor DB, kembalikan Parent Chunk-nya (dedupe, urut relevansi)."""
    vectorstore: Any
    docstore: Any
    k: int = 10
    child_k: int = 40

    def _get_relevant_documents(self, query: str, *, run_manager=None) -> List[Document]:
        children = self.vectorstore.similarity_search(query, k=self.child_k)
        ids, seen = [], set()
        for c in children:
            pid = c.metadata["parent_id"]
            if pid not in seen:
                seen.add(pid)
                ids.append(pid)
            if len(ids) >= self.k:
                break
        return [p for p in self.docstore.mget(ids) if p is not None]

retriever = ParentChildRetriever(vectorstore=vectorstore, docstore=docstore, k=5)

query = "Kapan karyawan berhak mendapat uang lembur?"
relevant_docs = retriever.invoke(query)
for i, doc in enumerate(relevant_docs[:3], start=1):
    print(f"--- Dokumen {i} ---")
    print(f"Sitasi: {doc.metadata.get('citation', '-')}")
    print(doc.page_content[:600])
    print()

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

--- Dokumen 1 ---
Sitasi: PP Nomor 35 Tahun 2021 halaman 22
turut atau lebih, meskipun Pengusaha membayar
Upah secara tepat waktu sesudah itu;
4. tidak melakukan kewajiban yang telah dijanjikan
kepada Pekerja/Buruh;
5. memerintahkan Pekerja/Buruh untuk
melaksanakan pekerjaan di luar yang
diperjanjikan; atau
6. memberikan pekerjaan yang membahayakan jiwa,
keselamatan, kesehatan, dan kesusilaan
Pekerja/Buruh sedangkan pekerjaan tersebut
tidak dicantumkan pada Perjanjian Kerja;
h. adanya putusan lembaga penyelesaian perselisihan
hubungan industrial yang menyatakan Pengusaha
tidak melakukan perbuatan sebagaimana dimaksud
pada huruf g terhadap permohonan ya

--- Dokumen 2 ---
Sitasi: PP Nomor 35 Tahun 2021 halaman 19
PRESIDEN
REPUELIK INDONESIA
-L9-
a. untuk ja- kerja lembur pertama sebesar
1,5 (satu koma lima) kali Upah sejam; dan
b. untuk setiap ja- kerja lembur berikutnya,
sebesar 2 (dua) kali Upah sejam.
(2) Perusahaan yang mempekerjakan Pekerja/Buruh
sebagaimana dimaksud pada ayat (1) 

Ensemble Retriever (BM25 + Vektor) & HyDE

In [6]:
try:
    from langchain.retrievers import EnsembleRetriever
except ImportError:
    from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever

ENSEMBLE_TOP_K = 10   # minimal 5 dokumen

bm25_retriever = BM25Retriever.from_documents(parent_docs)
bm25_retriever.k = ENSEMBLE_TOP_K
vector_retriever = ParentChildRetriever(vectorstore=vectorstore, docstore=docstore, k=ENSEMBLE_TOP_K)

ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.5, 0.5],
)

# ---------- Metadata filtering ----------
def infer_filter(query):
    """Deteksi 'PP 35 Tahun 2021' / 'UU Nomor 6' pada query -> filter metadata."""
    m = re.search(r"\b(UU|PP)\s*(?:Nomor|No\.?)?\s*(\d+)(?:\s*Tahun\s*(\d{4}))?", query, flags=re.IGNORECASE)
    if not m:
        return None
    flt = {"jenis": m.group(1).upper(), "nomor": int(m.group(2))}
    if m.group(3):
        flt["tahun"] = int(m.group(3))
    return flt

def apply_filter(docs, flt):
    if not flt:
        return docs
    kept = [d for d in docs if all(d.metadata.get(k) == v for k, v in flt.items())]
    if not kept:
        print("[filter] tidak ada dokumen cocok, filter diabaikan:", flt)
        return docs
    return kept

# ---------- HyDE ----------
HYDE_SYSTEM = ("Kamu pakar hukum ketenagakerjaan dan peraturan perundang-undangan Indonesia. "
               "Tulis jawaban hipotesis singkat (2-3 kalimat) seperti kutipan ketentuan peraturan yang relevan "
               "terhadap pertanyaan, dalam Bahasa Indonesia. Jangan gunakan tag <think>.")

def clean_hypothesis(t):
    c = re.sub(r"<think>.*?</think>", "", t, flags=re.DOTALL)
    c = re.sub(r"</?think>", "", c).strip()
    if len(c) < 20:
        c = re.sub(r"</?think>", "", t).strip()
    return c

def generate_hypotheses(query, n=2):
    outs = llm_generate(
        [{"role": "system", "content": HYDE_SYSTEM}, {"role": "user", "content": query}],
        max_new_tokens=160, temperature=0.8, n=n,
    )
    return [clean_hypothesis(o) for o in outs]

def hyde_ensemble_retrieve(query, metadata_filter=None, n_hypo=2):
    """Query asli + >=2 jawaban hipotesis -> Ensemble Retriever -> gabungkan & dedupe."""
    hypotheses = generate_hypotheses(query, n=n_hypo)
    candidates, seen = [], set()
    for q in [query] + hypotheses:
        for d in ensemble_retriever.invoke(q):
            if d.page_content not in seen:
                seen.add(d.page_content)
                candidates.append(d)
    return apply_filter(candidates, metadata_filter), hypotheses

# ---------- Uji ----------
test_q = "Apakah jika saya lembur saya akan dibayar lebih?"
cands, hypos = hyde_ensemble_retrieve(test_q)
for i, h in enumerate(hypos, 1):
    print(f"Hipotesis {i}: {h}\n")
print(f"Jumlah kandidat setelah HyDE + Ensemble: {len(cands)}")
for i, d in enumerate(cands[:5], 1):
    print(f"[{i}] {d.metadata['citation']} :: {d.page_content[:150].replace(chr(10), ' ')}...")

Both `max_new_tokens` (=160) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Hipotesis 1: Menurut Undang-Undang No. 5 Tahun 1999 tentang Ketenagakerjaan, jam lembur adalah jam kerja yang ditetapkan oleh karyawan untuk bekerja lebih dari jam kerja harian dan bulanan. Namun, untuk memastikan apakah Anda akan dibayar lebih, Anda harus mencari informasi lebih lanjut dari atasan Anda atau perwakilan HR yang berwenang. Mereka dapat memberikan panduan lebih spesifik mengenai kebijakan dan prosedur perusahaan mengenai jam lembur dan gaji.

Hipotesis 2: Menurut Undang-Undang Ketenagakerjaan, pemberian upah khusus bagi pekerja lembur adalah pilihan perusahaan. Pemberian upah khusus untuk pekerja lembur tidak diperlukan untuk pekerjaan lembur tetapi dapat menjadi praktik umum dan dapat memberikan pengakuan kepada pekerja lembur atas pengorbanan yang mereka buat untuk pekerjaan mereka. Namun, pekerja lembur tidak perlu menerima upah khusus untuk pekerjaan lembur.

Jumlah kandidat setelah HyDE + Ensemble: 41
[1] PP Nomor 35 Tahun 2021 halaman 19 :: PRESIDEN REPUELIK INDONES

Cross-Encoder Reranker & Conditional DuckDuckGo

In [7]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
try:
    from ddgs import DDGS
except ImportError:
    from duckduckgo_search import DDGS

RERANK_MODEL = "BAAI/bge-reranker-base"
RERANK_THRESHOLD = 0.25   # skor sigmoid 0-1; kalibrasi lewat uji di bawah
rr_tok = AutoTokenizer.from_pretrained(RERANK_MODEL)
rr_model = AutoModelForSequenceClassification.from_pretrained(RERANK_MODEL, torch_dtype=torch.float16).to("cuda").eval()

def rerank(query, docs, top_k=3, batch_size=8):
    scores = []
    for i in range(0, len(docs), batch_size):
        batch = docs[i:i + batch_size]
        enc = rr_tok([query] * len(batch), [d.page_content for d in batch],
                     padding=True, truncation="only_second", max_length=512, return_tensors="pt").to("cuda")
        with torch.no_grad():
            logits = rr_model(**enc).logits.view(-1).float()
        scores.extend(torch.sigmoid(logits).cpu().tolist())
    ranked = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)
    return ranked[:top_k]

def web_search(query, max_results=3):
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(f"{query} hukum Indonesia", region="id-id", max_results=max_results))
    except Exception as e:
        print("[DuckDuckGo] gagal:", e)
        return []
    return [Document(page_content=r.get("body", ""),
                     metadata={"source": r.get("href", ""), "title": r.get("title", "")}) for r in results]

def retrieve_context(query, metadata_filter=None):
    flt = metadata_filter or infer_filter(query)
    candidates, hypotheses = hyde_ensemble_retrieve(query, flt)
    ranked = rerank(query, candidates, top_k=3)
    top1 = ranked[0][1] if ranked else 0.0
    if top1 >= RERANK_THRESHOLD:                       # dokumen lokal relevan
        return {"mode": "local", "docs": [d for d, _ in ranked], "scores": [s for _, s in ranked],
                "top1": top1, "hypotheses": hypotheses}
    # Top-1 di bawah threshold -> ABAIKAN dokumen lokal, cari di internet
    return {"mode": "web", "docs": web_search(query), "scores": [], "top1": top1, "hypotheses": hypotheses}

# ---------- Uji & kalibrasi threshold ----------
for q in ["Apakah jika saya lembur saya akan dibayar lebih?",
          "Bagaimana resep rendang yang enak?"]:
    r = retrieve_context(q)
    print(f"\nQuery: {q}\n  Skor Top-1: {r['top1']:.3f} | threshold: {RERANK_THRESHOLD} | mode: {r['mode']}")
    for d in r["docs"]:
        print("  -", d.metadata.get("citation") or d.metadata.get("title"), "|", d.metadata.get("source", "")[:60])

config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Both `max_new_tokens` (=160) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=160) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Query: Apakah jika saya lembur saya akan dibayar lebih?
  Skor Top-1: 0.548 | threshold: 0.25 | mode: local
  - UU Nomor 6 Tahun 2023 halaman 559 | UU Nomor 6 Tahun 2023.pdf
  - PP Nomor 35 Tahun 2021 halaman 17 | PP Nomor 35 Tahun 2021.pdf
  - PP Nomor 35 Tahun 2021 halaman 27 | PP Nomor 35 Tahun 2021.pdf

Query: Bagaimana resep rendang yang enak?
  Skor Top-1: 0.000 | threshold: 0.25 | mode: web
  - Resep Rendang dengan Bumbu Halus Sederhana, Cukup 7 Bahan Bumbu | https://www.kompas.com/food/read/2023/04/27/210500775/resep-
  - Rendang Daging Istimewa Asli Khas Padang - Resep | ResepKoki | https://resepkoki.id/resep/resep-rendang-daging-sapi-padang/
  - Resep Rendang Daging Enak dan Praktis dengan Bahan yang Mudah Dicari ... | https://mediaedukasi.id/resep-rendang-daging-enak-dan-prakti


Chatbot Interaktif & Render Markdown

In [8]:
from IPython.display import Markdown, display

RAG_SYSTEM = ("Kamu adalah asisten AI internal Tim Legal yang menjawab pertanyaan hukum secara akurat dan "
              "dapat dipertanggungjawabkan, hanya berdasarkan konteks dokumen yang diberikan. Jangan mengarang pasal.")

RAG_USER_TEMPLATE = """Konteks dokumen:
{context}

Pertanyaan: {question}

Instruksi:
1. Tulis proses berpikir penalaran hukum langkah demi langkah di dalam tag <think>...</think>, sebutkan peraturan/pasal yang relevan dari konteks.
2. Setelah </think>, tulis jawaban final yang logis, ringkas, dan berbahasa Indonesia.
3. Jika konteks tidak memuat jawabannya, katakan bahwa informasinya tidak ditemukan."""

def format_context(res):
    parts = []
    for i, d in enumerate(res["docs"], 1):
        label = d.metadata.get("citation") or d.metadata.get("title") or "web"
        parts.append(f"[{i}] ({label})\n{d.page_content[:1200]}")
    return "\n\n".join(parts) if parts else "(tidak ada konteks ditemukan)"

def split_reasoning(text):
    if "</think>" in text:
        think, final = text.split("</think>", 1)
        return think.replace("<think>", "").strip(), final.strip()
    if "<think>" in text:
        return text.split("<think>", 1)[1].strip(), "_(Jawaban terpotong sebelum selesai. Coba ulangi pertanyaan.)_"
    return "", text.strip()

def ask(question, metadata_filter=None):
    res = retrieve_context(question, metadata_filter)
    prompt = RAG_USER_TEMPLATE.format(context=format_context(res), question=question)
    raw = llm_generate([{"role": "system", "content": RAG_SYSTEM}, {"role": "user", "content": prompt}],
                       max_new_tokens=768, temperature=0.3)[0]
    think, final = split_reasoning(raw)

    if res["mode"] == "local":
        src_note = f"Dokumen internal (skor relevansi Top-1: {res['top1']:.2f})"
        sources, seen = [], set()
        for d in res["docs"]:
            c = d.metadata["citation"]
            if c not in seen:
                seen.add(c)
                sources.append(f"- {c}")
    else:
        src_note = f"Dokumen lokal tidak relevan (skor Top-1: {res['top1']:.2f}), jawaban memakai pencarian internet"
        sources = [f"- [{d.metadata.get('title') or d.metadata['source']}]({d.metadata['source']})" for d in res["docs"]] \
                  or ["- (tidak ada hasil pencarian)"]

    quoted = think.replace("\n", "\n> ") if think else "_(model tidak menampilkan penalaran)_"
    md = (f"### 🧠 Proses Penalaran\n> {quoted}\n\n"
          f"### ✅ Jawaban\n{final}\n\n"
          f"### 📚 Sumber\n*{src_note}*\n\n" + "\n".join(sources))
    display(Markdown(md))

# Uji wajib (agar output tersimpan di notebook)
ask("Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. Apakah saya berhak dapat uang lembur?")

# Interactive loop
print("\nChatbot Tim Legal siap. Ketik 'keluar' untuk berhenti.")
while True:
    try:
        q = input("\nPertanyaan: ").strip()
    except (EOFError, KeyboardInterrupt):
        break
    if q.lower() in {"keluar", "exit", "quit"}:
        print("Sesi selesai.")
        break
    if q:
        ask(q)

Both `max_new_tokens` (=160) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=768) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### 🧠 Proses Penalaran
> 1. Pertama-tama, saya harus menentukan apakah saya berhak mendapatkan uang lembur. Menurut Pasal 79 UU Nomor 6 Tahun 2023, pengusaha wajib memberi waktu istirahat dan cuti kepada pekerja/buruh, termasuk waktu istirahat lembur. Namun, ada ketentuan yang harus dipenuhi untuk mendapatkan uang lembur. Menurut Pasal 79 ayat (1) huruf a, waktu istirahat lembur harus diberikan kepada pekerja/buruh paling sedikit meliputi istirahat antara jam kerja, paling sedikit setengah jam setelah bekerja selama 4 jam terus-menerus, dan waktu istirahat tersebut tidak termasuk jam kerja. Jadi, saya harus bekerja selama 4 jam terus-menerus dan kemudian beristirahat selama setengah jam sebelum saya dapat mendapatkan uang lembur.
> 2. Saya harus memeriksa apakah saya bekerja selama 4 jam terus-menerus. Jika saya bekerja selama 4 jam terus-menerus, saya dapat mendapatkan uang lembur.
> 3. Saya harus memeriksa apakah saya beristirahat selama setengah jam setelah bekerja selama 4 jam terus-menerus. Jika saya beristirahat selama setengah jam setelah bekerja selama 4 jam terus-menerus, saya dapat mendapatkan uang lembur.
> 4. Saya harus memeriksa apakah saya bekerja selama 4 jam terus-menerus dan beristirahat selama setengah jam setelah bekerja selama 4 jam terus-menerus. Jika saya bekerja selama 4 jam terus-menerus dan beristirahat selama setengah jam setelah bekerja selama 4 jam terus-menerus, saya dapat mendapatkan uang lembur.
> 5. Saya harus memeriksa apakah saya bekerja selama 4 jam terus-menerus, beristirahat selama setengah jam setelah bekerja selama 4 jam terus-menerus, dan apakah saya bekerja selama 3 jam lembur. Jika saya bekerja selama 4 jam terus-menerus, beristirahat selama setengah jam setelah bekerja selama 4 jam terus-menerus, dan bekerja selama 3 jam lembur, saya dapat mendapatkan uang lembur.
> 6. Saya harus memeriksa apakah saya bekerja selama 4 jam terus-menerus, beristirahat selama setengah jam setelah bekerja selama 4 jam terus-menerus, dan bekerja selama 3 jam lembur. Jika saya bekerja selama 4 jam terus-menerus, beristirahat selama setengah jam setelah bekerja selama 4 jam terus-menerus, dan bekerja selama 3 jam lembur, saya dapat mendapatkan uang lembur.
> 7. Saya harus memeriksa apakah saya bekerja selama 4 jam terus-menerus, beristirahat selama setengah jam setelah bekerja selama 4 jam terus-menerus, dan bekerja selama 3 jam

### ✅ Jawaban
_(Jawaban terpotong sebelum selesai. Coba ulangi pertanyaan.)_

### 📚 Sumber
*Dokumen internal (skor relevansi Top-1: 0.60)*

- PP Nomor 35 Tahun 2021 halaman 25
- UU Nomor 6 Tahun 2023 halaman 559
- PP Nomor 35 Tahun 2021 halaman 3


Chatbot Tim Legal siap. Ketik 'keluar' untuk berhenti.



Pertanyaan:  Apa saja hak yang diterima oleh karyawan jika mereka terkena Pemutusan Hubungan Kerja (PHK) karena perusahaan melakukan efisiensi?


Both `max_new_tokens` (=160) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=768) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### 🧠 Proses Penalaran
> _(model tidak menampilkan penalaran)_

### ✅ Jawaban
Pemutusan Hubungan Kerja (PHK) adalah keputusan pengusaha untuk memutus hubungan kerja dengan seorang pekerja atau buruh. Pekerja/buruh yang terkena PHK berhak atas beberapa hak, termasuk:

1. Uang pesangon: Pekerja/buruh berhak atas uang pesangon sebesar 0,5 kali ketentuan Pasal 40 ayat (2), yang berarti mereka akan menerima 50% dari upah mereka selama 3 bulan terakhir sebelum PHK.

2. Uang penghargaan masa kerja: Pekerja/buruh berhak atas uang penghargaan masa kerja sebesar 1 kali ketentuan Pasal 40 ayat (3), yang berarti mereka akan menerima 100% dari upah mereka selama 3 bulan terakhir sebelum PHK.

3. Uang penggantian hak: Pekerja/buruh berhak atas uang penggantian hak sesuai ketentuan Pasal 40 ayat (4), yang berarti mereka akan menerima upah mereka selama 3 bulan terakhir sebelum PHK.

4. Uang penggantian hak sesuai ketentuan Pasal 40 ayat (4): Pekerja/buruh berhak atas uang penggantian hak sesuai ketentuan Pasal 40 ayat (4), yang berarti mereka akan menerima upah mereka selama 3 bulan terakhir sebelum PHK.

5. Uang penggantian hak sesuai ketentuan Pasal 40 ayat (4): Pekerja/buruh berhak atas uang penggantian hak sesuai ketentuan Pasal 40 ayat (4), yang berarti mereka akan menerima upah mereka selama 3 bulan terakhir sebelum PHK.

6. Uang penggantian hak sesuai ketentuan Pasal 40 ayat (4): Pekerja/buruh berhak atas uang penggantian hak sesuai ketentuan Pasal 40 ayat (4), yang berarti mereka akan menerima upah mereka selama 3 bulan terakhir sebelum PHK.

7. Uang penggantian hak sesuai ketentuan Pasal 40 ayat (4): Pekerja/buruh berhak atas uang penggantian hak sesuai ketentuan Pasal 40 ayat (4), yang berarti mereka akan menerima upah mereka selama 3 bulan terakhir sebelum PHK.

8. Uang penggantian hak sesuai ketentuan Pasal 40 ayat (4): Pekerja/buruh berhak atas uang penggantian hak sesuai ketentuan Pasal 40 ayat (4), yang berarti mereka akan menerima upah mereka selama 3 bulan terakhir sebelum PHK.

9. Uang penggantian hak sesuai ketentuan Pasal 40 ayat (4): Pekerja/buruh berhak atas uang penggantian hak sesuai ketentuan Pasal 40 ayat (4), yang berarti mereka akan menerima upah mereka selama 3 bulan terakhir sebelum PHK.

10. Uang penggantian hak sesuai ketentuan Pasal 40 ayat (4): Pekerja/buruh berhak atas uang penggantian hak

### 📚 Sumber
*Dokumen internal (skor relevansi Top-1: 1.00)*

- PP Nomor 35 Tahun 2021 halaman 28
- PP Nomor 35 Tahun 2021 halaman 29
- UU Nomor 6 Tahun 2023 halaman 579


Pertanyaan:  keluar


Sesi selesai.


requirements.txt

In [9]:
from importlib.metadata import version, PackageNotFoundError

PACKAGES = ["unsloth", "unsloth_zoo", "trl", "peft", "transformers", "datasets", "accelerate",
            "bitsandbytes", "torch", "wandb", "langchain", "langchain-core", "langchain-community",
            "langchain-text-splitters", "langchain-huggingface", "faiss-cpu", "rank-bm25", "pypdf",
            "sentence-transformers", "ddgs", "duckduckgo-search", "ipython"]

lines = []
for p in PACKAGES:
    try:
        lines.append(f"{p}=={version(p)}")
    except PackageNotFoundError:
        print("Dilewati (tidak terpasang):", p)

with open("requirements.txt", "w") as f:
    f.write("\n".join(lines) + "\n")
print(open("requirements.txt").read())

# Alternatif lengkap: !pip freeze > requirements_full.txt

unsloth==2026.10.7
unsloth_zoo==2026.10.7
trl==1.15.0
peft==0.20.0
transformers==5.16.1
datasets==4.8.5
accelerate==1.14.0
bitsandbytes==0.50.2
torch==2.11.0+cu128
wandb==0.28.1
langchain==0.3.30
langchain-core==0.3.86
langchain-community==0.3.31
langchain-text-splitters==0.3.11
langchain-huggingface==0.3.1
faiss-cpu==1.15.1
rank-bm25==0.2.2
pypdf==6.19.0
sentence-transformers==5.7.0
ddgs==9.16.0
duckduckgo-search==8.1.1
ipython==7.34.0

